In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lombscargle
from sklearn.decomposition import PCA
from collections import Counter
from picoscenes import Picoscenes

# =========================================================
# 設定
# =========================================================
FS = 100

LOWCUT = 0.1
HIGHCUT = 1.0

N_SELECT = 256
PCA_COMPONENTS = 3

BPM_MIN = 6
BPM_MAX = 40

# =========================================================
# CSI読み込み
# =========================================================
def load_csi_matrix_for_nufft(csi_file):
    """
    PicoScenesのCSIを読み込み、最も多いCSI長(target_len)のデータだけを厳選。
    同時に、厳選されたパケットと完全にインデックスが一致するタイムスタンプ(ナノ秒)を抽出して返す。
    (Lomb-Scargle等の不均一解析にそのまま投入するための関数)
    """
    frames = Picoscenes(csi_file, 0, 0)

    csi_list = []
    lengths = []
    timestamps_ns = []  # タイムスタンプを一時的に格納する配列

    for frame in frames.raw:
        csi = frame.get("CSI")
        if csi is None:
            continue

        rx_basic = frame.get("RxSBasic")
        if rx_basic is None or "systemns" not in rx_basic:
            continue

        complex_csi = np.array(csi.get("CSI")).flatten()

        csi_list.append(complex_csi)
        lengths.append(len(complex_csi))
        timestamps_ns.append(rx_basic["systemns"])

    if len(csi_list) == 0:
        raise ValueError("CSIデータ、またはタイムスタンプが見つかりません。")

    # 1. 最も多いCSI長(サブキャリア数)の割り出し
    length_counter = Counter(lengths)
    target_len = length_counter.most_common(1)[0][0]

    # 🌟 2. ここでCSIとタイムスタンプを同じインデックス条件で厳選（同期）させる
    valid_csi = []
    valid_time_ns = []
    for csi, length, t_ns in zip(csi_list, lengths, timestamps_ns):
        if length == target_len:
            valid_csi.append(csi)
            valid_time_ns.append(t_ns)

    # NumPy配列化
    raw_csi_matrix = np.array(valid_csi)
    raw_time_array = np.array(valid_time_ns)

    # 🌟 行列とタイムスタンプの「2つの変数」をセットで返却する
    return raw_csi_matrix, raw_time_array

# =========================================================
# サブキャリア選択
# =========================================================
def select_subcarriers_by_snr(amplitude_all, fs, lowcut, highcut, n_select):
    n_samples, n_subcarriers = amplitude_all.shape
    n_select = min(n_select, n_subcarriers)

    # 平均を引いてFFT
    amplitude_centered = amplitude_all - np.mean(amplitude_all, axis=0)

    freqs = np.fft.rfftfreq(n_samples, d=1 / fs)
    fft_values = np.fft.rfft(amplitude_centered, axis=0)
    power = np.abs(fft_values) ** 2

    # 呼吸帯域とノイズ帯域
    breath_band = (freqs >= lowcut) & (freqs <= highcut)
    noise_band = (freqs > highcut) & (freqs <= fs / 2)

    breath_power = np.sum(power[breath_band, :], axis=0)
    noise_power = np.sum(power[noise_band, :], axis=0)

    snr = breath_power / (noise_power + 1e-12)

    selected_indices = np.argsort(snr)[-n_select:]
    selected_indices = np.sort(selected_indices)

    selected_amplitude = amplitude_all[:, selected_indices]

    # 選択されたサブキャリアの呼吸帯域ピーク周波数を確認
    band_freqs = freqs[breath_band]
    selected_peak_freqs = []

    for idx in selected_indices:
        band_power = power[breath_band, idx]
        peak_freq = band_freqs[np.argmax(band_power)]
        selected_peak_freqs.append(peak_freq)

    selected_peak_freqs = np.array(selected_peak_freqs)

    return selected_amplitude, selected_indices, snr

def estimate_breathing_rate_by_lombscargle_matrix(csi_file, raw_csi_matrix, raw_timestamps_ns, bpm_min=6, bpm_max=50, n_components=3):
    """
    時間領域PCA × Lomb-Scargle法による呼吸数推定
    """
    # 1. タイムスタンプを「秒」に正規化 (最初のパケットを0秒とする)
    t = (raw_timestamps_ns - raw_timestamps_ns[0]) / 1e9
    total_duration = t[-1]
    
    # 2. 全サブキャリアの振幅を抽出 (形状: [パケット数, サブキャリア数])
    amplitude_all = np.abs(raw_csi_matrix)
    n_samples, n_subcarriers = amplitude_all.shape
    
    # 直流(DC)成分を各サブキャリアごとに除去
    amplitude_centered = amplitude_all - np.mean(amplitude_all, axis=0)

    frames = Picoscenes(csi_file, 0, 0)
    # 🌟【追加】実際のサンプリング周波数(fs)をタイムスタンプから動的に計算
    # load_csi_matrix内で計算しているログのロジックを利用します
    total_ns = frames.raw[len(frames.raw)-1]['RxSBasic']['systemns'] - frames.raw[0]['RxSBasic']['systemns']
    total_seconds = total_ns / 1e9

    actual_fs = len(frames.raw) / total_seconds
    
    dt = 1 / actual_fs

    # サブキャリア選択
    selected_amplitude, selected_indices, snr = select_subcarriers_by_snr(
        amplitude_all=amplitude_centered,
        fs=actual_fs,
        lowcut=LOWCUT,
        highcut=HIGHCUT,
        n_select=N_SELECT
    )
    
    # 3. 【高速化】時間領域での主成分分析 (Time-domain PCA)
    # 形状: [パケット数, サブキャリア数] -> [パケット数, n_components(3)] に圧縮
    pca_time = PCA(n_components=n_components)
    time_pcs = pca_time.fit_transform(selected_amplitude)

    # ---------------------------------------------------------
    # 【Lomb-Scargleアルゴリズム適用】
    # パケットロスの発生により不均一間隔となったタイムスタンプデータ(t)に対して、
    # リサンプリングを行わずに直接周波数スペクトルを計算します。
    # ---------------------------------------------------------
    
    # 4. 探索する周波数軸 (Hz) と角周波数 (omega) を定義
    # 呼吸周波数の探索範囲 (0.05 Hz 〜 1.5 Hz) と対応する角周波数(omega)を生成
    freqs_hz = np.linspace(0.05, 1.5, 1000) 
    omegas = 2 * np.pi * freqs_hz
    freqs_bpm = freqs_hz * 60
    
    # 5. 主成分分析(PCA)によって抽出された各時間主成分に対してLomb-Scargle法を実行
    # 形状: [周波数軸の長さ(1000), サブキャリア数] の行列が生成されます
    print(f"--- 全 {n_subcarriers} サブキャリアの不均一周波数解析を実行中 ---")

    pgram_matrix = []
    for idx in range(n_components):
        # Lomb-Scargleピリオドグラムの算出
        # precenter=True: 平均値を差し引く
        # normalize=True: パワーを正規化する
        # floating_mean=True: フローティング平均モデルでオフセット誤差を軽減
        pgram = lombscargle(t, time_pcs[:, idx], omegas, precenter=True, normalize=True, floating_mean=True)
        pgram_matrix.append(pgram)
        
    # 配列化して転置 (形状を [周波数の細かさ, サブキャリア数] に揃える)
    pgram_matrix = np.array(pgram_matrix).T

    
    # 6. 呼吸ターゲット帯域内での評価とベストな主成分の選択
    valid_mask = (freqs_bpm >= bpm_min) & (freqs_bpm <= bpm_max)
    valid_freqs_bpm = freqs_bpm[valid_mask]
    
    best_pc_idx = 0
    max_peak_power = -1
    estimated_bpm = 0.0
    
    for i in range(n_components):
        pc_pgram = pgram_matrix[:, i]
        valid_pc_pgram = pc_pgram[valid_mask]
        
        peak_idx = np.argmax(valid_pc_pgram)
        peak_power = valid_pc_pgram[peak_idx]
        
        if peak_power > max_peak_power:
            max_peak_power = peak_power
            best_pc_idx = i
            estimated_bpm = valid_freqs_bpm[peak_idx]

    print(f"--- 信号融合・推定結果 ---")
    print(f"採用された時間主成分: Time-domain PC{best_pc_idx + 1}")
    print(f"当該PCの分散説明率: {pca_time.explained_variance_ratio_[best_pc_idx]:.4f}")
    print(f"【Time PCA + Lomb-Scargle】推定呼吸数: {estimated_bpm:.2f} bpm")
    print(f"----------------------------------------")
    
    # 7. ピリオドグラムの可視化
    plt.figure(figsize=(12, 5))
    for i in range(n_components):
        alpha = 1.0 if i == best_pc_idx else 0.4
        lw = 2.5 if i == best_pc_idx else 1.2
        label = f"Time PC{i+1} Spectrum" + (" (Selected)" if i == best_pc_idx else "")
        plt.plot(freqs_bpm, pgram_matrix[:, i], label=label, alpha=alpha, linewidth=lw)
        
    plt.axvline(estimated_bpm, color="red", linestyle="--", label=f"Estimated Peak: {estimated_bpm:.2f} bpm")
    plt.axvspan(bpm_min, bpm_max, alpha=0.1, color="green", label="Breathing Range")
    plt.title("Lomb-Scargle Periodograms of Time-domain PCs (Fast Mode)")
    plt.xlabel("Frequency [bpm]")
    plt.ylabel("Power (Normalized)")
    plt.xlim(0, 60)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    return estimated_bpm

def main():
    csi_file = "rx_2_260707_161059bpm15.csi"
    raw_csi_matrix, raw_time_array = load_csi_matrix_for_nufft(csi_file)

    estimated_bpm = estimate_breathing_rate_by_lombscargle_matrix(csi_file, raw_csi_matrix, raw_time_array)

    print(f"推定呼吸数: {estimated_bpm:.2f} bpm")
    return estimated_bpm

if __name__ == "__main__":
    main()